# 01 – Data cleaning: NHANES 2021–2023

Cleans the four NHANES files used in this project and saves one prepared CSV per file in `data/processed/`.

| File | Content |
|---|---|
| `DEMO_L` | demographics: age, sex, ethnicity, education, income |
| `BMX_L` | body measurements: weight, height, BMI, waist, hip |
| `DIQ_L` | diabetes questionnaire: diagnosis reported |
| `GHB_L` | HbA1c blood test (the target) |

Only adults (20 years and older) are kept.

## 1.1 Load the four NHANES files

Part A – Demographics (`DEMO_L`) starts here.

In [1]:
import pandas as pd
from pathlib import Path

data_dir = Path("../data")

# load the four NHANES files (SAS transport format)
demo = pd.read_sas(data_dir / "DEMO_L.xpt")
bmx = pd.read_sas(data_dir / "BMX_L.xpt")
diq = pd.read_sas(data_dir / "DIQ_L.xpt")
ghb = pd.read_sas(data_dir / "GHB_L.xpt")

print("DEMO:", demo.shape)
print("BMX:", bmx.shape)
print("DIQ:", diq.shape)
print("GHB:", ghb.shape)

DEMO: (11933, 27)
BMX: (8860, 22)
DIQ: (11744, 9)
GHB: (7199, 3)


In [2]:
demo.head()

,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDRETH1,RIDRETH3,RIDEXMON,RIDEXAGM,...,DMDHRGND,DMDHRAGZ,DMDHREDZ,DMDHRMAZ,DMDHSEDZ,WTINT2YR,WTMEC2YR,SDMVSTRA,SDMVPSU,INDFMPIR
0,130378.0,12.0,2.0,1.0,43.0,NaN,5.0,6.0,2.0,NaN,...,NaN,NaN,NaN,NaN,NaN,50055.450807,54374.463898,173.0,2.0,5.00
1,130379.0,12.0,2.0,1.0,66.0,NaN,3.0,3.0,2.0,NaN,...,NaN,NaN,NaN,NaN,NaN,29087.450605,34084.721548,173.0,2.0,5.00
2,130380.0,12.0,2.0,2.0,44.0,NaN,2.0,2.0,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,80062.674301,81196.277992,174.0,1.0,1.41
3,130381.0,12.0,2.0,2.0,5.0,NaN,5.0,7.0,1.0,71.0,...,2.0,2.0,2.0,3.0,NaN,38807.268902,55698.607106,182.0,2.0,1.53
4,130382.0,12.0,2.0,1.0,2.0,NaN,3.0,3.0,2.0,34.0,...,2.0,2.0,3.0,1.0,2.0,30607.519774,36434.146346,182.0,2.0,3.60


In [3]:
demo.isna().sum()

SEQN            0
SDDSRVYR        0
RIDSTATR        0
RIAGENDR        0
RIDAGEYR        0
RIDAGEMN    11556
RIDRETH1        0
RIDRETH3        0
RIDEXMON     3073
RIDEXAGM     9146
DMQMILIZ     3632
DMDBORN4       19
DMDYRUSR    10058
DMDEDUC2     4139
DMDMARTZ     4141
RIDEXPRG    10430
DMDHHSIZ        0
DMDHRGND     7818
DMDHRAGZ     7809
DMDHREDZ     8187
DMDHRMAZ     7913
DMDHSEDZ     9806
WTINT2YR        0
WTMEC2YR        0
SDMVSTRA        0
SDMVPSU         0
INDFMPIR     2041
dtype: int64

## 1.2 Check duplicates

In [4]:
demo.duplicated().sum()

np.int64(0)

In [5]:
demo_original=demo.copy(deep=True)
# Check the dimensions
print("Original dataset:", demo.shape)
print("Backup copy:", demo_original.shape)


Original dataset: (11933, 27)
Backup copy: (11933, 27)


## 1.3 Keep only the needed columns

In [6]:
# creating small working dataset

keep_columns=[
    "SEQN",
    "RIDAGEYR",
    "RIAGENDR",
    "RIDRETH3",
    "DMDEDUC2",
    "INDFMPIR",
    "RIDSTATR",
    "WTMEC2YR",
    "SDMVSTRA",
    "SDMVPSU"
]

# creating a new dataframe with selected columns

demo_clean= demo_original[keep_columns].copy()

# verify the results
print(demo_original.shape)
print(demo_clean.shape)

(11933, 27)
(11933, 10)


## 1.4 Check missing values

In [7]:
demo_clean.isna().sum()

SEQN           0
RIDAGEYR       0
RIAGENDR       0
RIDRETH3       0
DMDEDUC2    4139
INDFMPIR    2041
RIDSTATR       0
WTMEC2YR       0
SDMVSTRA       0
SDMVPSU        0
dtype: int64

In [8]:
demo_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 11933 entries, 0 to 11932
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   SEQN      11933 non-null  float64
 1   RIDAGEYR  11933 non-null  float64
 2   RIAGENDR  11933 non-null  float64
 3   RIDRETH3  11933 non-null  float64
 4   DMDEDUC2  7794 non-null   float64
 5   INDFMPIR  9892 non-null   float64
 6   RIDSTATR  11933 non-null  float64
 7   WTMEC2YR  11933 non-null  float64
 8   SDMVSTRA  11933 non-null  float64
 9   SDMVPSU   11933 non-null  float64
dtypes: float64(10)
memory usage: 932.4 KB


In [9]:
demo_clean.head()

,SEQN,RIDAGEYR,RIAGENDR,RIDRETH3,DMDEDUC2,INDFMPIR,RIDSTATR,WTMEC2YR,SDMVSTRA,SDMVPSU
0,130378.0,43.0,1.0,6.0,5.0,5.00,2.0,54374.463898,173.0,2.0
1,130379.0,66.0,1.0,3.0,5.0,5.00,2.0,34084.721548,173.0,2.0
2,130380.0,44.0,2.0,2.0,3.0,1.41,2.0,81196.277992,174.0,1.0
3,130381.0,5.0,2.0,7.0,NaN,1.53,2.0,55698.607106,182.0,2.0
4,130382.0,2.0,1.0,3.0,NaN,3.60,2.0,36434.146346,182.0,2.0


## 1.5 Missing education by age, and select adults (20+)

**Finding:** almost all missing education values belong to people under 20, so we keep adults only.

In [10]:
demo_clean['DMDEDUC2'].describe()

count    7794.000000
mean        3.804978
std         1.153750
min         1.000000
25%         3.000000
50%         4.000000
75%         5.000000
max         9.000000
Name: DMDEDUC2, dtype: float64

In [11]:
demo_clean['DMDEDUC2'].value_counts(dropna=False).sort_index()

DMDEDUC2
1.0     373
2.0     666
3.0    1749
4.0    2370
5.0    2625
9.0      11
NaN    4139
Name: count, dtype: int64

In [12]:

# Missing education among participants younger than 20
print("Missing education among people under 20:")
print(
    demo_clean.loc[
        demo_clean["RIDAGEYR"] < 20,
        "DMDEDUC2"
    ].isna().sum()
)

# Missing education among adults aged 20+
print("Missing education among adults aged 20+:")
print(
    demo_clean.loc[
        demo_clean["RIDAGEYR"] >= 20,
        "DMDEDUC2"
    ].isna().sum()
)


Missing education among people under 20:
4124
Missing education among adults aged 20+:
15


In [13]:

# Create adult-only working dataset

demo_adults = demo_clean.loc[
    demo_clean["RIDAGEYR"] >= 20
].copy()

# Compare datasets
print("Original demographic data:", demo_original.shape)
print("Selected demographic data:", demo_clean.shape)
print("Adult-only data:", demo_adults.shape)

# Check missing values among adults
print("\nMissing values among adults:")
print(demo_adults.isna().sum())


Original demographic data: (11933, 27)
Selected demographic data: (11933, 10)
Adult-only data: (7809, 10)

Missing values among adults:
SEQN           0
RIDAGEYR       0
RIAGENDR       0
RIDRETH3       0
DMDEDUC2      15
INDFMPIR    1320
RIDSTATR       0
WTMEC2YR       0
SDMVSTRA       0
SDMVPSU        0
dtype: int64


In [14]:
demo_adults.isna().sum()

SEQN           0
RIDAGEYR       0
RIAGENDR       0
RIDRETH3       0
DMDEDUC2      15
INDFMPIR    1320
RIDSTATR       0
WTMEC2YR       0
SDMVSTRA       0
SDMVPSU        0
dtype: int64

In [15]:
demo_adults['INDFMPIR'].describe()

count    6.489000e+03
mean     2.900643e+00
std      1.659736e+00
min      5.397605e-79
25%      1.400000e+00
50%      2.790000e+00
75%      4.970000e+00
max      5.000000e+00
Name: INDFMPIR, dtype: float64

In [16]:

# Explore income values without modifying the data

print("Values equal to 5:")
print((demo_adults["INDFMPIR"] == 5).sum())

print("Values below 0:")
print((demo_adults["INDFMPIR"] < 0).sum())

print("Values between 0 and 1:")
print((demo_adults["INDFMPIR"] < 1).sum())


Values equal to 5:
1616
Values below 0:
0
Values between 0 and 1:
1037


In [17]:

# Check education categories among adults

demo_adults["DMDEDUC2"].value_counts(
    dropna=False
).sort_index()


DMDEDUC2
1.0     373
2.0     666
3.0    1749
4.0    2370
5.0    2625
9.0      11
NaN      15
Name: count, dtype: int64

In [18]:

import numpy as np

# Create a new working copy
demo_prepared = demo_adults.copy()

# Convert special education codes to missing
demo_prepared["DMDEDUC2"] = (
    demo_prepared["DMDEDUC2"].replace({
        7: np.nan,   # Refused
        9: np.nan    # Don't know
    })
)

# Compare missing values before and after
print("Before cleaning:",
      demo_adults["DMDEDUC2"].isna().sum())

print("After cleaning:",
      demo_prepared["DMDEDUC2"].isna().sum())


Before cleaning: 15
After cleaning: 26


In [19]:
demo_prepared.info()

<class 'pandas.DataFrame'>
Index: 7809 entries, 0 to 11932
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   SEQN      7809 non-null   float64
 1   RIDAGEYR  7809 non-null   float64
 2   RIAGENDR  7809 non-null   float64
 3   RIDRETH3  7809 non-null   float64
 4   DMDEDUC2  7783 non-null   float64
 5   INDFMPIR  6489 non-null   float64
 6   RIDSTATR  7809 non-null   float64
 7   WTMEC2YR  7809 non-null   float64
 8   SDMVSTRA  7809 non-null   float64
 9   SDMVPSU   7809 non-null   float64
dtypes: float64(10)
memory usage: 671.1 KB


## 1.6 Rename the variables

The original NHANES names are hard to read, so they get clear names.

In [20]:

# Rename demographic variables as their original name are not easy to understand

rename_columns = {
    "RIDAGEYR": "age_years",
    "RIAGENDR": "gender_code",
    "RIDRETH3": "race_ethnicity_code",
    "DMDEDUC2": "education_code",
    "INDFMPIR": "income_poverty_ratio",
    "RIDSTATR": "exam_status_code",
    "WTMEC2YR": "mec_exam_weight",
    "SDMVSTRA": "survey_stratum",
    "SDMVPSU": "survey_psu"
}

# Create a new DataFrame with renamed columns
demo_final = demo_prepared.rename(
    columns=rename_columns
)

# Check the new variable names
print(demo_final.columns.tolist())

# Check the dimensions
print("Prepared dataset:", demo_prepared.shape)
print("Renamed dataset:", demo_final.shape)

display(demo_final.head())


['SEQN', 'age_years', 'gender_code', 'race_ethnicity_code', 'education_code', 'income_poverty_ratio', 'exam_status_code', 'mec_exam_weight', 'survey_stratum', 'survey_psu']
Prepared dataset: (7809, 10)
Renamed dataset: (7809, 10)


,SEQN,age_years,gender_code,race_ethnicity_code,education_code,income_poverty_ratio,exam_status_code,mec_exam_weight,survey_stratum,survey_psu
0,130378.0,43.0,1.0,6.0,5.0,5.00,2.0,5.437446e+04,173.0,2.0
1,130379.0,66.0,1.0,3.0,5.0,5.00,2.0,3.408472e+04,173.0,2.0
2,130380.0,44.0,2.0,2.0,3.0,1.41,2.0,8.119628e+04,174.0,1.0
6,130384.0,43.0,1.0,1.0,2.0,0.63,1.0,5.397605e-79,179.0,2.0
7,130385.0,65.0,2.0,3.0,3.0,5.00,1.0,5.397605e-79,187.0,2.0


## 1.7 Save the prepared demographic dataset

In [21]:

# after all processing saving final dataset
from pathlib import Path
processed_folder=Path(
    '../data/processed'
)
processed_folder.mkdir(parents=True, exist_ok=True)

# saved the prepared demographic dataset
demo_final.to_csv(
    processed_folder/'demo_prepared.csv',
    index=False
)
print("Dataset saved successfully!")
print("Shape:", demo_final.shape)

Dataset saved successfully!
Shape: (7809, 10)


In [22]:

# Check that the CSV was created and both saved and current dat have same data
file_path = processed_folder / "demo_prepared.csv"

print("File exists:", file_path.exists())

# Read the file back
demo_check = pd.read_csv(file_path)

# Compare dimensions
print("Before saving:", demo_final.shape)
print("After loading:", demo_check.shape)


File exists: True
Before saving: (7809, 10)
After loading: (7809, 10)


## 1.8 Part B – Body measurements (`BMX_L`): keep a copy of the original

In [23]:
# step 1: preserve original copy of BMX_L dataset

bmx_original= bmx.copy(deep=True)

# checking dataset dimension 

bmx_original.shape

# displaying all columns names

bmx_original.columns.to_list()

['SEQN',
 'BMDSTATS',
 'BMXWT',
 'BMIWT',
 'BMXRECUM',
 'BMIRECUM',
 'BMXHEAD',
 'BMIHEAD',
 'BMXHT',
 'BMIHT',
 'BMXBMI',
 'BMDBMIC',
 'BMXLEG',
 'BMILEG',
 'BMXARML',
 'BMIARML',
 'BMXARMC',
 'BMIARMC',
 'BMXWAIST',
 'BMIWAIST',
 'BMXHIP',
 'BMIHIP']

In [24]:
bmx_original.head()

,SEQN,BMDSTATS,BMXWT,BMIWT,BMXRECUM,BMIRECUM,BMXHEAD,BMIHEAD,BMXHT,BMIHT,...,BMXLEG,BMILEG,BMXARML,BMIARML,BMXARMC,BMIARMC,BMXWAIST,BMIWAIST,BMXHIP,BMIHIP
0,130378.0,1.0,86.9,NaN,NaN,NaN,NaN,NaN,179.5,NaN,...,42.8,NaN,42.0,NaN,35.7,NaN,98.3,NaN,102.9,NaN
1,130379.0,1.0,101.8,NaN,NaN,NaN,NaN,NaN,174.2,NaN,...,38.5,NaN,38.7,NaN,33.7,NaN,114.7,NaN,112.4,NaN
2,130380.0,1.0,69.4,NaN,NaN,NaN,NaN,NaN,152.9,NaN,...,38.5,NaN,35.5,NaN,36.3,NaN,93.5,NaN,98.0,NaN
3,130381.0,1.0,34.3,NaN,NaN,NaN,NaN,NaN,120.1,NaN,...,NaN,NaN,25.4,NaN,23.4,NaN,70.4,NaN,NaN,NaN
4,130382.0,3.0,13.6,NaN,NaN,1.0,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,1.0,NaN,1.0,NaN,1.0,NaN,NaN


## 1.9 Check missing values

In [25]:
bmx_original.isna().sum()

SEQN           0
BMDSTATS       0
BMXWT        106
BMIWT       8515
BMXRECUM    8406
BMIRECUM    8842
BMXHEAD     8790
BMIHEAD     8860
BMXHT        361
BMIHT       8726
BMXBMI       389
BMDBMIC     6368
BMXLEG      1525
BMILEG      8464
BMXARML      292
BMIARML     8660
BMXARMC      298
BMIARMC     8655
BMXWAIST     670
BMIWAIST    8513
BMXHIP      2084
BMIHIP      8499
dtype: int64

## 1.10 Keep only the needed columns

In [26]:
# selecting variables and dropping unwanted

# variables we want to keep

keep_columns = [
    "SEQN",
    "BMXWT",
    "BMXHT",
    "BMXBMI",
    "BMXWAIST",
    "BMXHIP"
]

# identify variables we dont need

drop_columns=bmx_original.columns.difference(keep_columns) # This compares all the original column names with our list and identifies the ones that are not included.

# Create a new working dataset

bmx_clean= bmx_original.drop(columns=drop_columns)

# verify the results

print("Original dataset:", bmx_original.shape)
print("Working dataset:", bmx_clean.shape)

Original dataset: (8860, 22)
Working dataset: (8860, 6)


In [27]:
bmx_clean.head()

,SEQN,BMXWT,BMXHT,BMXBMI,BMXWAIST,BMXHIP
0,130378.0,86.9,179.5,27.0,98.3,102.9
1,130379.0,101.8,174.2,33.5,114.7,112.4
2,130380.0,69.4,152.9,29.7,93.5,98.0
3,130381.0,34.3,120.1,23.8,70.4,NaN
4,130382.0,13.6,NaN,NaN,NaN,NaN


## 1.11 Check duplicates

In [28]:
bmx_clean.duplicated().sum()

np.int64(0)

In [29]:

# Check duplicate participant IDs
bmx_clean["SEQN"].duplicated().sum()


np.int64(0)

## 1.12 Missing values in the selected columns

In [30]:
bmx_clean.isna().sum()

SEQN           0
BMXWT        106
BMXHT        361
BMXBMI       389
BMXWAIST     670
BMXHIP      2084
dtype: int64

## 1.13 Select adults (hip circumference has many missing values in children)

In [31]:
# add age to body measurement data for exploration

bmx_age= bmx_clean.merge(
    demo_final[['SEQN', 'age_years']],
    on='SEQN',
    how='left'
)

# checking missing values among adults

bmx_adults=bmx_age[bmx_age['age_years'] >=20].copy()

print(bmx_clean.shape)
print(bmx_adults.shape)

(8860, 6)
(6064, 7)


In [32]:

# Check missing values in adult body measurements

bmx_adults.isna().sum()


SEQN           0
BMXWT         81
BMXHT         67
BMXBMI        94
BMXWAIST     302
BMXHIP       363
age_years      0
dtype: int64

## 1.14 Explore body measurement values among adults

In [33]:

bmx_adults[["BMXWT", "BMXHT", "BMXBMI", "BMXWAIST", "BMXHIP"]].describe()


,BMXWT,BMXHT,BMXBMI,BMXWAIST,BMXHIP
count,5983.000000,5997.000000,5970.000000,5762.000000,5701.000000
mean,83.292006,166.897982,29.827035,101.087817,107.869777
std,22.374309,10.062665,7.358598,16.898478,14.355361
min,27.900000,133.000000,11.100000,60.000000,74.500000
25%,67.550000,159.600000,24.700000,88.900000,98.000000
50%,79.900000,166.500000,28.500000,99.500000,105.000000
75%,95.400000,174.200000,33.600000,111.400000,114.900000
max,248.200000,200.700000,74.800000,187.000000,187.100000


In [34]:
# Automatically select body measurement variables # alternative way for describe 
bmx_adults.filter(like="BMX").describe() #  alternative way for describe if not able to write variable that are in abbreviated forms

,BMXWT,BMXHT,BMXBMI,BMXWAIST,BMXHIP
count,5983.000000,5997.000000,5970.000000,5762.000000,5701.000000
mean,83.292006,166.897982,29.827035,101.087817,107.869777
std,22.374309,10.062665,7.358598,16.898478,14.355361
min,27.900000,133.000000,11.100000,60.000000,74.500000
25%,67.550000,159.600000,24.700000,88.900000,98.000000
50%,79.900000,166.500000,28.500000,99.500000,105.000000
75%,95.400000,174.200000,33.600000,111.400000,114.900000
max,248.200000,200.700000,74.800000,187.000000,187.100000


## 1.15 Check unusually low and high BMI values

In [35]:
# Check the 5 lowest BMI values
bmx_adults.sort_values("BMXBMI").head(5)

,SEQN,BMXWT,BMXHT,BMXBMI,BMXWAIST,BMXHIP,age_years
8585,141941.0,27.9,158.4,11.1,62.8,74.5,66.0
5766,138163.0,32.6,152.2,14.1,60.0,74.7,31.0
8334,141592.0,41.4,166.9,14.9,69.3,NaN,33.0
8301,141553.0,41.7,167.2,14.9,63.1,84.5,24.0
6398,139004.0,46.2,174.3,15.2,73.7,78.8,59.0


In [36]:

# Check the 5 highest BMI values
bmx_adults.sort_values("BMXBMI", ascending=False).head(5)


,SEQN,BMXWT,BMXHT,BMXBMI,BMXWAIST,BMXHIP,age_years
3894,135637.0,185.2,157.3,74.8,NaN,NaN,41.0
1225,132014.0,179.5,160.2,69.9,169.0,176.2,54.0
3614,135246.0,222.7,179.5,69.1,NaN,NaN,41.0
3003,134413.0,248.2,189.8,68.9,187.0,180.4,33.0
2954,134348.0,192.1,167.5,68.5,170.0,187.1,61.0


In [37]:

# Check the lowest BMI value manually

weight = 27.9
height = 158.4 / 100

bmi_calculated = weight / (height ** 2)

print("Calculated BMI:", round(bmi_calculated, 1))


Calculated BMI: 11.1


## 1.16 Check BMI consistency (recorded vs calculated)

In [38]:

# Check BMI consistency

# Create a copy to protect our existing dataset
bmx_check = bmx_adults.copy()

# Calculate BMI using weight and height
bmx_check["bmi_calculated"] = (
    bmx_check["BMXWT"] /
    ((bmx_check["BMXHT"] / 100) ** 2)
)

# Calculate the difference from recorded BMI
bmx_check["bmi_difference"] = (
    bmx_check["BMXBMI"] - bmx_check["bmi_calculated"]
).abs()

# Explore the differences
bmx_check["bmi_difference"].describe()


count    5970.000000
mean        0.024536
std         0.014488
min         0.000000
25%         0.011862
50%         0.024732
75%         0.036744
max         0.049965
Name: bmi_difference, dtype: float64

## 1.17 Rename the body measurement variables

In [39]:
# Create a separate working copy without the temporary age column
bmx_final=bmx_adults.drop(columns=['age_years'])
# Rename columns
bmx_final = bmx_final.rename(columns={
    "BMXWT": "weight_kg",
    "BMXHT": "height_cm",
    "BMXBMI": "bmi",
    "BMXWAIST": "waist_cm",
    "BMXHIP": "hip_cm"
})

# Check the result
print(bmx_final.shape)
display(bmx_final.head())


(6064, 6)


,SEQN,weight_kg,height_cm,bmi,waist_cm,hip_cm
0,130378.0,86.9,179.5,27.0,98.3,102.9
1,130379.0,101.8,174.2,33.5,114.7,112.4
2,130380.0,69.4,152.9,29.7,93.5,98.0
5,130386.0,90.6,173.3,30.2,106.1,110.6
6,130387.0,103.5,155.9,42.6,122.0,148.9


## 1.18 Save the prepared body measurement dataset

In [40]:
#  save the prepared body measurement dataset

bmx_final.to_csv(
    processed_folder/'bmx_prepared.csv',
    index=False
)

# verify that the file exists

# Verify that the file exists
print((processed_folder / "bmx_prepared.csv").exists())

# Check the dataset dimensions
print(bmx_final.shape)


True
(6064, 6)


## 1.19 Part C – Diabetes questionnaire (`DIQ_L`): first look

In [41]:
# creating copy of original dataset

diq_original= diq.copy()

# check rows and col

print(diq_original.shape)

# check all variables names

print(diq_original.columns.tolist())

(11744, 9)
['SEQN', 'DIQ010', 'DID040', 'DIQ160', 'DIQ180', 'DIQ050', 'DID060', 'DIQ060U', 'DIQ070']


In [42]:
diq_original.head()

,SEQN,DIQ010,DID040,DIQ160,DIQ180,DIQ050,DID060,DIQ060U,DIQ070
0,130378.0,2.0,NaN,2.0,2.0,NaN,NaN,NaN,NaN
1,130379.0,2.0,NaN,2.0,1.0,NaN,NaN,NaN,NaN
2,130380.0,1.0,35.0,NaN,NaN,2.0,NaN,NaN,1.0
3,130381.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,130382.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [43]:
diq_original.isna().sum()

SEQN           0
DIQ010         4
DID040     10663
DIQ160      3722
DIQ180      3440
DIQ050     10663
DID060     11401
DIQ060U    11412
DIQ070      9463
dtype: int64

## 1.20 Keep only the needed columns

In [44]:
#  keep only prediabetes, diabets and participants id

diq_clean= diq_original[["SEQN", "DIQ010", "DIQ160"]].copy()

# check the results

print(diq_original.shape)
print(diq_clean.shape)



(11744, 9)
(11744, 3)


## 1.21 Understand the diabetes answers

In [45]:

# Check diabetes questionnaire response categories

print("Diabetes responses:")
print(diq_clean["DIQ010"].value_counts(dropna=False).sort_index())

print("Prediabetes responses:")
print(diq_clean["DIQ160"].value_counts(dropna=False).sort_index())


Diabetes responses:
DIQ010
1.0     1081
2.0    10371
3.0      284
9.0        4
NaN        4
Name: count, dtype: int64
Prediabetes responses:
DIQ160
1.0     918
2.0    7089
9.0      15
NaN    3722
Name: count, dtype: int64


In [46]:

# Display diabetes responses with meaningful labels

diq_clean["DIQ010"].map({
    1: "Yes",
    2: "No",
    3: "Borderline",
    7: "Refused",
    9: "Don't know"
}).value_counts(dropna=False)


DIQ010
No            10371
Yes            1081
Borderline      284
NaN               4
Don't know        4
Name: count, dtype: int64

## 1.22 Check duplicates

In [47]:
#— Check duplicates

diq_clean.duplicated().sum()

np.int64(0)

In [48]:
# Check duplicated participant IDs
diq_clean["SEQN"].duplicated().sum()


np.int64(0)

## 1.23 Clean special missing-value codes (7 = refused, 9 = don't know)

In [49]:

import numpy as np

# Create a new copy to protect diq_clean
diq_prepared = diq_clean.copy()

# Replace special missing-value codes
diq_prepared["DIQ010"] = diq_prepared["DIQ010"].replace({
    7: np.nan,
    9: np.nan
})

diq_prepared["DIQ160"] = diq_prepared["DIQ160"].replace({
    7: np.nan,
    9: np.nan
})

# Check missing values after cleaning
diq_prepared.isna().sum()


SEQN         0
DIQ010       8
DIQ160    3737
dtype: int64

## 1.24 Rename the variables

In [50]:

# Rename diabetes questionnaire variables

diq_final = diq_prepared.rename(columns={
    "DIQ010": "diabetes_reported",
    "DIQ160": "prediabetes_reported"
})

# Check the results
print(diq_final.shape)
print(diq_final.columns.tolist())

display(diq_final.head())


(11744, 3)
['SEQN', 'diabetes_reported', 'prediabetes_reported']


,SEQN,diabetes_reported,prediabetes_reported
0,130378.0,2.0,2.0
1,130379.0,2.0,2.0
2,130380.0,1.0,NaN
3,130381.0,2.0,NaN
4,130382.0,2.0,NaN


## 1.25 Select adults (20+)

In [51]:

# Keep only adult participants in DIQ

diq_adults = diq_final[
    diq_final["SEQN"].isin(demo_final["SEQN"])
].copy()

# Compare dataset dimensions
print("Before:", diq_final.shape)
print("After:", diq_adults.shape)

display(diq_adults.head())


Before: (11744, 3)
After: (7809, 3)


,SEQN,diabetes_reported,prediabetes_reported
0,130378.0,2.0,2.0
1,130379.0,2.0,2.0
2,130380.0,1.0,NaN
6,130384.0,2.0,2.0
7,130385.0,2.0,1.0


## 1.26 Save the prepared diabetes questionnaire dataset

In [52]:
diq_adults.to_csv(
    processed_folder/ 'diq_prepared.csv',
    index=False
)

In [53]:
# Verify the saved file
print((processed_folder / "diq_prepared.csv").exists())
print(diq_adults.shape)

True
(7809, 3)


## 1.27 Part D – HbA1c blood test (`GHB_L`): first look

In [54]:
#  loading dataset

ghb=pd.read_sas(data_dir / "GHB_L.xpt"
                )

# preserve original dataset

ghb_original= ghb.copy()

# check shape

print(ghb_original.shape)

# Check variables names

print(ghb_original.head())

(7199, 3)
       SEQN       WTPH2YR  LBXGH
0  130378.0  56042.129410    5.6
1  130379.0  37435.705647    5.6
2  130380.0  85328.844519    6.2
3  130386.0  44526.214135    5.1
4  130387.0  22746.296353    5.9


## 1.28 Check duplicates and missing values

In [55]:

# Check duplicated rows
print("Duplicated rows:", ghb_original.duplicated().sum())

# Check duplicated participant IDs
print("Duplicated SEQN:", ghb_original["SEQN"].duplicated().sum())

# Check missing values
print("\nMissing values:")
print(ghb_original.isna().sum())


Duplicated rows: 0
Duplicated SEQN: 0

Missing values:
SEQN         0
WTPH2YR      0
LBXGH      484
dtype: int64


## 1.29 Select adults (20+)

In [56]:

# Select adults aged 20 years and above

ghb_adults = ghb_original[
    ghb_original["SEQN"].isin(demo_final["SEQN"])
].copy()

# Compare dimensions
print("Before:", ghb_original.shape)
print("After:", ghb_adults.shape)

# Check missing values among adults
print(ghb_adults.isna().sum())

# Display first five rows
display(ghb_adults.head())


Before: (7199, 3)
After: (6064, 3)
SEQN         0
WTPH2YR      0
LBXGH      297
dtype: int64


,SEQN,WTPH2YR,LBXGH
0,130378.0,56042.129410,5.6
1,130379.0,37435.705647,5.6
2,130380.0,85328.844519,6.2
3,130386.0,44526.214135,5.1
4,130387.0,22746.296353,5.9


## 1.30 Explore HbA1c values

In [57]:

# Explore HbA1c values among adults

ghb_adults["LBXGH"].describe()


count    5767.000000
mean        5.788174
std         1.112402
min         3.200000
25%         5.200000
50%         5.500000
75%         5.900000
max        17.100000
Name: LBXGH, dtype: float64

## 1.31 Keep adults with a measured HbA1c

In [58]:

# Keep adults with measured HbA1c

ghb_valid = ghb_adults.dropna(subset=["LBXGH"]).copy()

# Compare dimensions
print("Before:", ghb_adults.shape)
print("After:", ghb_valid.shape)

# Check missing values
print(ghb_valid.isna().sum())


Before: (6064, 3)
After: (5767, 3)
SEQN       0
WTPH2YR    0
LBXGH      0
dtype: int64


## 1.32 Select and rename the needed variables

In [59]:

# Select only the required variables
ghb_final = ghb_valid[["SEQN", "LBXGH"]].copy()

# Rename HbA1c variable
ghb_final = ghb_final.rename(columns={
    "LBXGH": "hba1c"
})

# Check results
print(ghb_final.shape)
print(ghb_final.isna().sum())

display(ghb_final.head())


(5767, 2)
SEQN     0
hba1c    0
dtype: int64


,SEQN,hba1c
0,130378.0,5.6
1,130379.0,5.6
2,130380.0,6.2
3,130386.0,5.1
4,130387.0,5.9


## 1.33 Save the prepared HbA1c dataset

In [60]:
## Save the prepare dataset

ghb_final.to_csv(
    processed_folder/ 'ghb_prepared.csv',
    index=False
)

#  verify the saved file

print((processed_folder/'ghb_prepared.csv').exists())
print(ghb_final.shape)

True
(5767, 2)
